<center>
    <h1>Probabilistic models taken by Storm</h1>
    <h2>Step 1: Modeling</h2>
    <br>
    <br>
    <img src="images/storm_logo.png" width="500px"/>
    <h3>Matthias Volk</h3>
</center>

<div align="right">Press <em>spacebar</em> to navigate</div>


# Setup

- We will use the Storm ecosystem, especially [Stormvogel](https://github.com/stormchecker/stormvogel)
- Material at [https://github.com/stormchecker/stormpyter/tree/master/tutorial_orchard](https://github.com/stormchecker/stormpyter/tree/master/tutorial_orchard)
- Try online: [try.stormchecker.org](https://try.stormchecker.org/)
  <div>
  <img src="images/qrcode.png" width="200px"/><img src="images/qrcode_material.png" width="200px"/></div>

In [ ]:
import stormvogel

In [ ]:
bird = stormvogel.show_bird()

In [ ]:
# Skip

# The Orchard model
- *Orchard* (*Obstgarten*, *Le verger*, *Boomgaard*, *El Frutal*, *Første Frukthag*, ...) is a simple children's game
- will use *First Orchard* game ([rules](https://cdn.hff.de/image/upload/v1/22/30/28/14/22302814.pdf))
<center>
    <img src="https://www.kinderchaos-familienblog.de/wp-content/uploads/2020/01/Obstgarten-Kinderspiel-scaled.jpg" width="30%"/>
</center>

([Credit](https://www.kinderchaos-familienblog.de/gesellschaftsspiele/obstgarten/))

## Rules
- different types of fruit (apples 🍏, pears 🍐, cherries 🍒, plums 🍇)
- tree per fruit with 4 pieces
- raven reaches orchard in 5 steps

**Goal**: collect all fruits before the raven reaches the orchard

### Round
Throw dice and
- **Color**  🍏🍐🍒🍇: pick fruit of corresponding color from tree
- **Basket** 🧺: choose fruit
- **Raven** 🐦‍⬛: move raven one step

# Stormvogel model
## Data structures

In [ ]:
# Imports
from enum import Enum
from copy import deepcopy

# General data structures
class Fruit(Enum):
    APPLE = "🍏"
    PEAR = "🍐"
    CHERRY = "🍒"
    PLUM = "🍇" # Use grape as there is no Emoji for plum

    def __str__(self):
        return self.value

class DiceOutcome(Enum):
    FRUIT = "🍉"
    BASKET = "🧺"
    RAVEN = "🐦‍⬛"

    def __str__(self):
        return self.value

class GameState(Enum):
    NOT_ENDED = 0
    PLAYERS_WON = 1
    RAVEN_WON = 2

In [ ]:
# Skip

## Game state
- states need to encode the status of the game<br>
$\to$ What information is needed?

In [ ]:
# Main class for the orchard game
class Orchard(stormvogel.bird.State):
    def __init__(self, fruit_types, num_fruits, raven_distance):
        self.trees = {fruit: num_fruits for fruit in fruit_types}
        self.raven = raven_distance
        self.dice = None

    def game_state(self):
        if all(self.is_tree_empty(tree) for tree in self.trees.keys()):
            assert self.raven > 0
            return GameState.PLAYERS_WON
        elif self.raven == 0:
            return GameState.RAVEN_WON
        else:
            return GameState.NOT_ENDED

    def is_tree_empty(self, tree):
        return self.trees[tree] == 0

    def pick_fruit(self, fruit):
        if self.trees[fruit] > 0:
            self.trees[fruit] -= 1

    def next_round(self):
        self.dice = None
        
    def move_raven(self):
        self.raven -= 1

    def __hash__(self):
        trees = [hash((f,n)) for f,n in self.trees.items()]
        return hash((tuple(trees), self.raven, self.dice))

    def label(self):
        if self.dice is None:
            # Output game state
            return str(self)
        else:
            if self.dice[0] == DiceOutcome.FRUIT:
                return "🎲" + str(self.dice[1])
            else:
                return "🎲" + str(self.dice[0])

    def __str__(self):
        s = ", ".join(["{}{}".format(n, f) for f, n in self.trees.items()])
        s += ", {}←{}".format(self.raven, DiceOutcome.RAVEN)
        return s

In [ ]:
# Skip

# Stormvogel model
## Game state
- states need to encode the status of the game
- states are considered equal if their Python objects are equal (hash)<br>
  $\to$ information to distinguish behavior must be stored
- only store *relevant* information<br>
  $\to$ otherwise state space might blow up


# MDP in Stormvogel

- use *bird* API of Stormvogel to define model
- need to specify all MDP ingredients: $\mathcal{M} = (S, s_0, Act, \mathbf{P}, AP, L)$
- need to provide:
  - `init`: the inital state $s_0$
  - `available_actions`: the defined actions $Act(s)$ per state
  - `delta`: the transition function $\mathbf{P}$
  - `labels`: the labeling function $L$
- *bird* iteratively builds the state space starting from initial state

# Orchard in Stormvogel
## Initial state $s_0$

In [ ]:
# Initialize the game
init_small = Orchard([Fruit.APPLE, Fruit.CHERRY],
                     num_fruits=2,
                     raven_distance=2)

In [ ]:
print(init_small)

In [ ]:
# Skip

## Available (defined) actions

In [ ]:
# Define available actions
def available_actions(state):
    if state.game_state() != GameState.NOT_ENDED:
        return ["gameEnded"]
    if state.dice is None:
        return ["nextRound"]
    if state.dice[0] == DiceOutcome.FRUIT:
        return ["pick{}".format(state.dice[1].name)]
    if state.dice[0] == DiceOutcome.BASKET:
        available_fruits = []
        # Choice over available fruits
        for fruit in state.trees.keys():
            if not state.is_tree_empty(fruit):
                available_fruits.append(fruit)
        return ["choose{}".format(fruit.name) for fruit in available_fruits]
    if state.dice[0] == DiceOutcome.RAVEN:
        return ["moveRaven"]
    assert False

In [ ]:
# Skip

## Transition function $\mathbf{P}$

In [ ]:
# The transition function
def delta(state, action):
    if state.game_state() != GameState.NOT_ENDED:
        assert action == "gameEnded"
        # Game has ended -> self loop
        return [(1, state)]

    if state.dice is None:
        assert action == "nextRound"
        # Player throws dice and considers outcomes
        outcomes = []
        # Probability of fair dice throw over
        # each fruit type + 1 basket + 1 raven
        fair_dice_prob = 1 / (len(state.trees.keys()) + 2)
        
        # 1. Dice shows fruit
        for fruit in state.trees.keys():
            next_state = deepcopy(state)
            next_state.dice = DiceOutcome.FRUIT, fruit
            outcomes.append((fair_dice_prob, next_state))
            
        # 2. Dice shows basket
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.BASKET, None
        outcomes.append((fair_dice_prob, next_state))
       
        # 3. Dice shows raven
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.RAVEN, None
        outcomes.append((fair_dice_prob, next_state))

        assert sum([o[0] for o in outcomes]) == 1
        return outcomes

    elif state.dice[0] == DiceOutcome.FRUIT:
        assert action.startswith("pick")
        # Player picks specified fruit
        fruit = state.dice[1]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.BASKET:
        assert action.startswith("choose")
        # Player chooses fruit specified by action
        fruit = Fruit[action.removeprefix("choose")]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.RAVEN:
        assert action == "moveRaven"
        next_state = deepcopy(state)
        next_state.move_raven()
        next_state.next_round()
        return [(1, next_state)]

    assert False

In [ ]:
# Skip

## Labeling function $L$

In [ ]:
# Add labels for game state
def labels(state):
    labels = [state.label()]
    if state.game_state() == GameState.PLAYERS_WON:
        labels.append("PlayersWon")
    elif state.game_state() == GameState.RAVEN_WON:
        labels.append("RavenWon")
    return labels

In [ ]:
# Skip

## Reward function $R$

In [ ]:
# Reward function
def rewards(state, action):
    if state.game_state() == GameState.NOT_ENDED:
        if state.dice is None:
            return {"rounds": 1}
    return {"rounds": 0}

In [ ]:
# Skip

## Build model

In [ ]:
# Build the simple orchard model
orchard_simple = stormvogel.bird.build_bird(
    modeltype=stormvogel.ModelType.MDP,
    init=init_small,
    available_actions=available_actions,
    delta=delta,
    labels=labels,
    rewards=rewards
)

print(orchard_simple.summary())

In [ ]:
# Skip

## Explore model

In [ ]:
orchard_explore = stormvogel.layout.DEFAULT()
orchard_explore.layout['misc']['explore'] = True

vis = stormvogel.show(orchard_simple, layout=orchard_explore)

In [ ]:
# Skip

## Visualize model

In [ ]:
# Visualize
vis = stormvogel.show(orchard_simple)

In [ ]:
# Skip

# Exercise

Let raven stop randomly:
- if dice throws raven 🐦‍⬛:
    - move raven one step forward with probability 0.8
    - do not move raven with probability 0.2

**$\to$ Task: Adapt your model and incorporate the extended raven movement**

[try.stormchecker.org](https://try.stormchecker.org/) $\to$ examples $\to$ tutorial $\to$ Orchard Modeling

## Updated transition function $\mathbf{P}$

In [ ]:
# The transition function
def delta_ex(state, action):
    if state.game_state() != GameState.NOT_ENDED:
        assert action == "gameEnded"
        # Game has ended -> self loop
        return [(1, state)]

    if state.dice is None:
        assert action == "nextRound"
        # Player throws dice and considers outcomes
        outcomes = []
        # Probability of fair dice throw over
        # each fruit type + 1 basket + 1 raven
        fair_dice_prob = 1 / (len(state.trees.keys()) + 2)
        
        # 1. Dice shows fruit
        for fruit in state.trees.keys():
            next_state = deepcopy(state)
            next_state.dice = DiceOutcome.FRUIT, fruit
            outcomes.append((fair_dice_prob, next_state))
            
        # 2. Dice shows basket
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.BASKET, None
        outcomes.append((fair_dice_prob, next_state))
       
        # 3. Dice shows raven
        next_state = deepcopy(state)
        next_state.dice = DiceOutcome.RAVEN, None
        outcomes.append((fair_dice_prob, next_state))

        assert sum([o[0] for o in outcomes]) == 1
        return outcomes

    elif state.dice[0] == DiceOutcome.FRUIT:
        assert action.startswith("pick")
        # Player picks specified fruit
        fruit = state.dice[1]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.BASKET:
        assert action.startswith("choose")
        # Player chooses fruit specified by action
        fruit = Fruit[action.removeprefix("choose")]
        next_state = deepcopy(state)
        next_state.pick_fruit(fruit)
        next_state.next_round()
        return [(1, next_state)]

    elif state.dice[0] == DiceOutcome.RAVEN:
        assert action == "moveRaven"
        outcomes = []
        # One step
        next_state = deepcopy(state)
        next_state.move_raven()
        next_state.next_round()
        outcomes.append((0.8, next_state))
        # Zero steps
        next_state = deepcopy(state)
        next_state.next_round()
        outcomes.append((0.2, next_state))
        assert sum([o[0] for o in outcomes])
        return outcomes

    assert False

In [ ]:
# skip

In [ ]:
# Build the simple orchard model
orchard_simple_ex = stormvogel.bird.build_bird(
    modeltype=stormvogel.ModelType.MDP,
    init=init_small,
    available_actions=available_actions,
    delta=delta_ex,
    labels=labels,
    rewards=rewards
)

print(orchard_simple_ex.summary())

In [ ]:
# skip

In [ ]:
vis = stormvogel.show(orchard_simple_ex)

In [ ]:
# skip